In [ ]:
from pathlib import Path
from typing import Sequence

import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

In [ ]:
path = "../"
path = Path(path).expanduser()
import sys
sys.path.insert(0, str(path))

In [ ]:
import decode
import decode.neuralfitter.inference.functional as infer_func
print(decode.__file__)
log = decode.generic.logging.get_logger(__name__)

%config InlineBackend.figure_format='retina'

In [ ]:
path_frames = "../data/Fig2a-SmallTest" # or change to your data path

path_frames = Path(path_frames).expanduser()

path_trafo = "../calibration/Fig2a-Pos0_MM_100ms_638i1_T700(100)_R685(70)_665LP_20nmstep_Z_1_up_noEM_trafo.mat"
path_trafo = Path(path_trafo).expanduser()

path_training = "../outputs/Fig2a-dual_color-actin_microtubes-separated_ch-2026-05-14_17-22-25-738645"
path_training = Path(path_training).expanduser()
path_cfg = path_training / "param_run.yaml"
path_ckpt = next((path_training).glob("*.ckpt"))

model_identifier = f"model_{path_ckpt.parent.stem}"

cfg = decode.io.param.load(path_cfg)
gain_correct = True

device = ["cuda:0"]

for s in cfg["Hardware"]["device"]:
    cfg["Hardware"]["device"][s] = device[0]
    



In [ ]:
cfg["Camera"][0]["specs"]

In [ ]:
# single dataset, single camera
path_subs = [sorted(path_frames.glob("*.tif"))[0].parent]

mode_camera = "rois"
bulk_load = False

path_subs

In [ ]:
def rearrange_frames(frames):
    # ###### Only needed when reference channel is down ######
    n = len(frames)
    x0 = frames[0]
    H, W = x0.shape  # 512, 226
    out = torch.empty((n, H, W), dtype=x0.dtype)

    half = H // 2
    idx = torch.cat([torch.arange(half, H), torch.arange(0, half)])

    chunk = 512
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        x = torch.stack([frames[i] for i in range(s, e)])
        x = x.index_select(1, idx.to(x.device))
        out[s:e].copy_(x)

    return out

In [ ]:

from decode.io.frames import TiffTensor 
for p in path_subs:
    pframe = sorted(p.glob("*.tif"))[0]
    frames = decode.io.frames.load_tif(pframe, auto_ome=True, memmap=True)
    print(f"Paths: \n{pframe}\nsize: {frames.size()}")

    if bulk_load:
        # ToDo: implement bulk for multi-camera
        frames = frames[:]
    
    # frames = rearrange_frames(frames) # only needed when reference channel is down
    
    frame_size = list(frames.size())
    frame_crop = [math.floor(frame_size[-2] /2 / 8) * 8, math.floor(frame_size[-1] / 8) * 8]
    print(f"Frame size: {frame_size} -> Crop: {frame_crop}")

    # load metadata
    # path_meta = sorted(pframe.parent.glob("*metadata.txt"))[0]
    # meta = decode.io.camera.load_metadata(path_meta)
    # shift = torch.as_tensor(meta.glob.get_roi_shift(gain_corrected=gain_correct) + [0]) 
    # print(f"ROI Shift: {shift}\nMetadata path: {path_meta}")

    path_ckpt = next((path_training).glob("*.ckpt"))
    em_out, logger = infer_func.infer(
        frames,
        frame_crop=frame_crop,
        cfg=cfg,
        model=path_ckpt,
        mode="multi",
        trafo=path_trafo,
        mode_camera=mode_camera,
        roi_shift=None, # shift[:2].tolist(),
        logger="debug",
        batch_size=4,
        num_workers=0,
        device = device,
    )

    em_save = em_out.clone()
    # em_save.xyz_px += shift

    path_out = Path("../results") / f"{pframe.stem}_decode_plex_fit_{model_identifier}.h5"
    path_out.parent.mkdir(parents=True, exist_ok=True)
    em_out.save(str(path_out))
    print(f"Saved to {path_out}")
    break

In [ ]:
cfg["Test"]["Transformation"]["Pos"]["glob"]["offset"]
# em_out = decode.EmitterSet.load("../results/MM_25ms_638i100_UVi20_focusloci30_1_MMStack_Default.ome_decode_plex_fit_model_2026-04-28_10-02-35-050133.h5")


In [ ]:
plt.subplots(figsize=(10, 10))
decode.plot.PlotFrame(frames[100] + frames[100].flip(-2)).plot()
plt.xlim(0, 208)
plt.show()
# decode.plot.PlotFrame(frames[100].flip(-2)).plot()

In [ ]:
subsample = 100

f, axs = plt.subplots(nrows=2, ncols=2, figsize=(12,  8))

_ = axs[0, 0].hist(em_out.prob[:].numpy(), bins=100)
axs[0, 0].set_title("Probability")

_ = axs[0, 1].hist(em_out.frame_ix.numpy(), bins=torch.arange(1000))
axs[0, 1].set_title("N Emitter over frame_ix")

em_rend = em_out[em_out.prob > 0.6]

for i, ax in enumerate(axs[1]):
    px_dist = decode.evaluation.predict_dist.px_pointer_dist(em_rend.xyz_px[::subsample, i], -0.5, 1.0)
    ax.hist(px_dist, bins=torch.linspace(-0.5, 0.5, 40).numpy())
axs[1, 0].set_title("px dist")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

em_p = em_out.clone()
em_p = em_p[em_p.prob > 0.6]
em_p = em_p[em_p.xyz_sig_lat_nm < 40]

# filtered_data = xyz_px[(xyz_px[:, 0] >= 40) & (xyz_px[:, 0] <= 120) &
#                        (xyz_px[:, 1] >= 40) & (xyz_px[:, 1] <= 120)]

for dim, label in zip([0, 1], ['x', 'y']):
    values = decode.evaluation.predict_dist.px_pointer_dist(em_p.xyz_px[:, dim], -0.5, 1.)
    ax.hist(values.numpy(), 
            bins=np.linspace(-0.5, 0.5, 20), 
            density=True, 
            # edgecolor='black',
            alpha=0.6, 
            label=label) 

ax.legend()  
ax.set_xlabel("subpixel position")
ax.set_ylabel("density")

plt.tight_layout()
plt.show()

## Quick Visualization of Localization Results

The parameters `xextent` and `yextent` control the rendered spatial region, while the `contrast` parameter adjusts the visualization contrast of the rendered image.

For the datasets used in the manuscript figures, we recommend the following settings:

- **Fig. 2a**:

    ```python
    xextent = yextent = (0, 200 * px_size[0])
    contrast = 1.
    ```

- **Fig. 2b**:

    ```python
    xextent = yextent = (0, 400 * px_size[0])
    contrast = 4.
    ```

These rendering settings are only intended for quick visualization of localization results within the notebook. For final analysis, quantitative evaluation, and interactive inspection, we recommend opening the provided .h5 localization files directly in SMAP.

In [ ]:
# rendering
import matplotlib as mpl
px_size = em_p.px_size

xextent = (0, 200 * px_size[0])
yextent = (0, 200 * px_size[1])
zextent = (-1000., 1000.)

renderer = decode.renderer.renderer.Renderer2D(
    xextent=xextent,
    yextent=xextent,
    colextent=zextent,
    px_size=10.,
    sigma_blur=10.,
    rel_clip=0.05,
    # contrast=4.,
    cmap="turbo",
)

img = renderer.forward(em_p, em_p.xyz[:, 2])


fig, ax = plt.subplots(figsize=(8, 8))

im = ax.imshow(img.permute(1, 0, 2))

plt.xticks([])
plt.yticks([])
# plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# Create an inset_axes for the colorbar next to each subplot
cmap = mpl.cm.turbo
# Adjust the [left, bottom, width, height] values as needed for your layout
cbar_ax = ax.inset_axes([1.02, 0., 0.05, 1.])

# Create a ScalarMappable with the turbo colormap and normalization
norm = mpl.colors.Normalize(vmin=zextent[0], vmax=zextent[1])
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Add the colorbar to the inset_axes
fig.colorbar(sm, cax=cbar_ax, fraction=0.046, pad=0.04)


plt.tight_layout()

In [ ]:
em_p = em_out[em_out.prob > 0.6]
em_p = em_p[em_p.xyz_sig_lat_nm < 60]
em_p = em_p[::2]

plt.scatter(em_p.phot[:, 0], em_p.phot[:, 1], s=0.1, color="k", marker=".", facecolors="k", edgecolors="none")

plt.xlabel("Channel 1")
plt.ylabel("Channel 2")
plt.title("Photon count")
plt.xscale("log")
plt.yscale("log")

plt.xscale("log")
plt.yscale("log")

plt.xlim(5e1, 5e4)
plt.ylim(5e1, 5e4)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# plot all inputs and raw outputs
f_ix = 3

f, axs = plt.subplots(ncols=4, nrows=3, figsize=(14, 8))
for i, ax in enumerate(axs.flatten()):
    x = logger.model_in[0][f_ix]
    if i >= x.shape[0]:
        ax.axis("off")
        continue
    im = ax.imshow(x[i])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)

plt.tight_layout()
plt.show()

f, axs = plt.subplots(ncols=4, nrows=4, figsize=(14, 12))
for i, ax in enumerate(axs.flatten()):
    x = logger.model_out[0][f_ix].cpu().numpy()
    if i >= x.shape[0]:
        ax.axis("off")
        continue

    im = ax.imshow(x[i])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)
plt.tight_layout()
plt.show()

In [ ]:
f, axs = plt.subplots(2, 4, figsize=(20, 10))

for i, r in enumerate([-1]):
    em_c = em_out

    axs[i, 0].hist(em_c.phot[:, 0].numpy(), bins=100, range=(1e2, 1e4))
    axs[i, 1].hist(em_c.phot[:, 1].numpy(), bins=100, range=(1e2, 1e4))
    axs[i, 2].hist(em_c.bg[:, 0].numpy(), bins=100)
    axs[i, 3].hist(em_c.bg[:, 1].numpy(), bins=100)

    if i == 0:
        axs[i, 0].set_title("Channel 1 phot")
        axs[i, 1].set_title("Channel 2 phot")
        axs[i, 2].set_title("Channel 1 bg")
        axs[i, 3].set_title("Channel 2 bg")